# m02 Multi-Vehicle RSSI Utility Search

Build exact-5 multi-vehicle subset examples from `data/interim`, then train the v3-style two-tower architecture on candidate utility functions. This notebook intentionally uses only normalized sensor coordinates and RSSI dB, so it is the RSSI-only ablation for the multi-vehicle setting.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from multi_vehicle_utility_search import build_eq5_rssi_static_examples

build_summary_df, build_paths_df = build_eq5_rssi_static_examples(PROJECT_ROOT)
display(build_summary_df)
display(build_paths_df)


Interim source: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\interim\20250815_0900_multi_vehicle
valid timestamps=6625, nodes=10, vehicles=['vehicle1', 'vehicle2'], exact subset size=5
rho=10.222 m
Saved processed artifact: m02_multi_vehicle_eq5_rssi_static
Elapsed: 0.7 min


,prefix,times,actions_per_time,examples,context_dim,action_dim,mean_contains_fraction_all_actions,both_rate_all_actions,elapsed_min
0,m02_multi_vehicle_eq5_rssi_static,6625,252,1669500,30,21,0.5,0.245493,0.74452


,artifact,path
0,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import SECOND_SWEEP_CANDIDATES, run_rssi_utility_search

summary_df, utility_results = run_rssi_utility_search(
    PROJECT_ROOT,
    model_tag="multi_vehicle_rssi_utility_search_eq5_second_sweep",
    max_epochs=150,
    patience=150,
    log_every=5,
    batch_size=8192,
    time_batch_size=256,
    utility_candidates=SECOND_SWEEP_CANDIDATES,
)

vehicle_cols = [col for col in summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(summary_df[[col for col in display_cols if col in summary_df.columns]])


Dense cache is missing requested utility targets; rebuilding with 12 utility matrices.
Building dense train cache from m02_multi_vehicle_eq5_rssi_static_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
utilities: ['softmin_d1_beta5', 'softmin_d1_beta10', 'sum_plus_softmin_beta10_lam05', 'sum_plus_softmin_beta10_lam10', 'harmonic_d1', 'sum_d1_minus_gap_lam025', 'top2_d1_alpha02']

=== softmin_d1_beta5 === Smooth min over per-vehicle d1 utilities with beta=5.
softmin_d1_beta5 ep 001/150* loss=0.04217 val_acc=0.601 both=0.326 one=0.550 gap=0

,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,softmin_d1_beta5,111,0.873585,0.760000,0.044528,0.869057,0.753208,0.231698,0.015094,0.000755,0.869434,0.868679,1.282264,0.267170,0.018317
1,sum_plus_softmin_beta10_lam10,150,0.877358,0.763019,0.024906,0.868302,0.761509,0.213585,0.024906,0.035472,0.886038,0.850566,1.265660,0.235472,0.032461
2,sum_d1_minus_gap_lam025,150,0.849811,0.713208,0.004528,0.866038,0.762264,0.207547,0.030189,0.038491,0.885283,0.846792,1.271698,0.233962,0.031175
3,top2_d1_alpha02,107,0.866038,0.761509,0.012830,0.866038,0.749434,0.233208,0.017358,0.059623,0.895849,0.836226,1.342641,0.325283,0.035205
4,sum_plus_softmin_beta10_lam05,118,0.857736,0.733585,0.015849,0.861887,0.738868,0.246038,0.015094,0.031698,0.877736,0.846038,1.303396,0.286792,0.027098
5,harmonic_d1,118,0.860377,0.737359,0.022642,0.848679,0.720755,0.255849,0.023396,0.030943,0.864151,0.833208,1.311698,0.286792,0.020767
6,softmin_d1_beta10,119,0.869811,0.756981,0.020377,0.846415,0.715472,0.261887,0.022642,0.033962,0.863396,0.829434,1.325283,0.299623,0.023069


In [6]:
# Re-run the original utility set for 150 epochs without overwriting earlier sweeps.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import UTILITY_CANDIDATES, run_rssi_utility_search

original_summary_df, original_utility_results = run_rssi_utility_search(
    PROJECT_ROOT,
    model_tag="multi_vehicle_rssi_utility_search_eq5_original_sweep_150ep",
    max_epochs=150,
    patience=150,
    log_every=5,
    batch_size=8192,
    time_batch_size=256,
    utility_candidates=UTILITY_CANDIDATES,
)

vehicle_cols = [col for col in original_summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(original_summary_df[[col for col in display_cols if col in original_summary_df.columns]])


PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
utilities: ['sum_full', 'sum_d1_only', 'min_d1_only', 'balanced_d1', 'balanced_full']

=== sum_full === u1_full + u2_full, where full uses d1/d2/d3 terms.
        sum_full ep 001/150* loss=0.50005 val_acc=0.570 both=0.289 one=0.562 gap=0.003 vehicle1=0.568 vehicle2=0.571 wrank=2.33 reg=0.1929 1s
        sum_full ep 002/150* loss=0.07388 val_acc=0.625 both=0.375 one=0.499 gap=0.008 vehicle1=0.620 vehicle2=0.629 wrank=2.14 reg=0.1754 1s
        sum_full ep 004/150* loss=0.03757 val_acc=0.640 both=0.435 one=0.410 gap=0.042 vehicle1=0.620 vehicle2=0.661 wrank=1.77 reg=0.1410 2s
        sum_full ep 005/150* loss=0.03351 val_acc=0.682 both=0.482 one=0.401 gap=0.008 vehicle1=

,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,balanced_full,132,0.869434,0.755472,0.016604,0.856226,0.723774,0.264906,0.011321,0.038491,0.875472,0.836981,1.384906,0.373585,0.034979
1,sum_d1_only,92,0.855472,0.735849,0.021887,0.852830,0.731321,0.243019,0.025660,0.019623,0.862642,0.843019,1.323019,0.295094,0.031356
2,balanced_d1,135,0.864528,0.744151,0.029434,0.850943,0.718491,0.264906,0.016604,0.033962,0.867925,0.833962,1.320000,0.302642,0.019514
3,sum_full,67,0.873585,0.758491,0.014340,0.850566,0.728302,0.244528,0.027170,0.048302,0.874717,0.826415,1.409057,0.379623,0.057036
4,min_d1_only,129,0.868302,0.753208,0.038491,0.837736,0.700377,0.274717,0.024906,0.037736,0.856604,0.818868,1.368302,0.341887,0.024701


In [7]:
# Softmin-utility parameter sweep: vary beta and lambda around the current best setting.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import run_rssi_utility_search

# Default setting is beta=10, lambda=1.0. In the labels, lam05=0.5, lam10=1.0, lam20=2.0.
SOFTMIN_PARAM_SWEEP_CANDIDATES = [
    {
        "name": "sum_plus_softmin_beta5_lam10",
        "column": "derived_sum_plus_softmin_beta5_lam10",
        "description": "mean(d1 utilities) + 1.0 * softmin_beta5(d1 utilities).",
        "kind": "sum_plus_softmin_d1",
        "beta": 5.0,
        "lam": 1.0,
    },
    {
        "name": "sum_plus_softmin_beta20_lam10",
        "column": "derived_sum_plus_softmin_beta20_lam10",
        "description": "mean(d1 utilities) + 1.0 * softmin_beta20(d1 utilities).",
        "kind": "sum_plus_softmin_d1",
        "beta": 20.0,
        "lam": 1.0,
    },
    {
        "name": "sum_plus_softmin_beta10_lam05",
        "column": "derived_sum_plus_softmin_beta10_lam05",
        "description": "mean(d1 utilities) + 0.5 * softmin_beta10(d1 utilities).",
        "kind": "sum_plus_softmin_d1",
        "beta": 10.0,
        "lam": 0.5,
    },
    {
        "name": "sum_plus_softmin_beta10_lam20",
        "column": "derived_sum_plus_softmin_beta10_lam20",
        "description": "mean(d1 utilities) + 2.0 * softmin_beta10(d1 utilities).",
        "kind": "sum_plus_softmin_d1",
        "beta": 10.0,
        "lam": 2.0,
    },
]

softmin_param_summary_df, softmin_param_results = run_rssi_utility_search(
    PROJECT_ROOT,
    model_tag="multi_vehicle_rssi_softmin_param_sweep_eq5_150ep",
    max_epochs=150,
    patience=150,
    log_every=5,
    batch_size=8192,
    time_batch_size=256,
    utility_candidates=SOFTMIN_PARAM_SWEEP_CANDIDATES,
)

vehicle_cols = [col for col in softmin_param_summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(softmin_param_summary_df[[col for col in display_cols if col in softmin_param_summary_df.columns]])

Dense cache is missing requested utility targets; rebuilding with 15 utility matrices.
Building dense train cache from m02_multi_vehicle_eq5_rssi_static_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
utilities: ['sum_plus_softmin_beta5_lam10', 'sum_plus_softmin_beta20_lam10', 'sum_plus_softmin_beta10_lam05', 'sum_plus_softmin_beta10_lam20']

=== sum_plus_softmin_beta5_lam10 === mean(d1 utilities) + 1.0 * softmin_beta5(d1 utilities).
sum_plus_softmin_beta5_lam10 ep 001/150* loss=0.20403 val_acc=0.611 both=0.340 one=0.543 gap=0.009 vehicl

,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,sum_plus_softmin_beta5_lam10,150,0.861509,0.733585,0.015849,0.871321,0.763774,0.215094,0.021132,0.021887,0.882264,0.860377,1.267170,0.245283,0.028979
1,sum_plus_softmin_beta20_lam10,150,0.877736,0.762264,0.031698,0.863396,0.751698,0.223396,0.024906,0.039245,0.883019,0.843774,1.283019,0.251321,0.037159
2,sum_plus_softmin_beta10_lam05,118,0.857736,0.733585,0.015849,0.861887,0.738868,0.246038,0.015094,0.031698,0.877736,0.846038,1.303396,0.286792,0.027098
3,sum_plus_softmin_beta10_lam20,113,0.851698,0.720000,0.002264,0.833585,0.701887,0.263396,0.034717,0.041509,0.854340,0.812830,1.391698,0.354717,0.067709


In [8]:
# Softmin-only utility sweep: choose beta for the clean multi-vehicle objective.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import run_rssi_utility_search

SOFTMIN_D1_BETA_SWEEP_CANDIDATES = []
for beta in [0.1, 1.0, 5.0, 20.0, 100.0]:
    beta_token = str(beta).replace(".", "p")
    SOFTMIN_D1_BETA_SWEEP_CANDIDATES.append(
        {
            "name": f"softmin_d1_beta{beta_token}",
            "column": f"derived_softmin_d1_beta{beta_token}",
            "description": f"Smooth min over per-vehicle d1 utilities with beta={beta}.",
            "kind": "softmin_d1",
            "beta": float(beta),
        }
    )

softmin_beta_summary_df, softmin_beta_results = run_rssi_utility_search(
    PROJECT_ROOT,
    model_tag="multi_vehicle_rssi_softmin_d1_beta_sweep_eq5_150ep",
    max_epochs=150,
    patience=150,
    log_every=5,
    batch_size=8192,
    time_batch_size=256,
    utility_candidates=SOFTMIN_D1_BETA_SWEEP_CANDIDATES,
)

vehicle_cols = [col for col in softmin_beta_summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(softmin_beta_summary_df[[col for col in display_cols if col in softmin_beta_summary_df.columns]])


Dense cache is missing requested utility targets; rebuilding with 17 utility matrices.
Building dense train cache from m02_multi_vehicle_eq5_rssi_static_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
utilities: ['softmin_d1_beta0p1', 'softmin_d1_beta1p0', 'softmin_d1_beta5p0', 'softmin_d1_beta20p0', 'softmin_d1_beta100p0']

=== softmin_d1_beta0p1 === Smooth min over per-vehicle d1 utilities with beta=0.1.
softmin_d1_beta0p1 ep 001/150* loss=0.04869 val_acc=0.577 both=0.295 one=0.565 gap=0.003 vehicle1=0.576 vehicle2=0.579 wrank=2.45 reg

,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,softmin_d1_beta5p0,111,0.873585,0.760000,0.044528,0.869057,0.753208,0.231698,0.015094,0.000755,0.869434,0.868679,1.282264,0.267170,0.018317
1,softmin_d1_beta100p0,129,0.869057,0.755472,0.041509,0.838491,0.704906,0.267170,0.027925,0.031698,0.854340,0.822641,1.359245,0.329811,0.024283
2,softmin_d1_beta1p0,142,0.863019,0.752453,0.003774,0.832830,0.711698,0.242264,0.046038,0.015849,0.840755,0.824906,1.329057,0.275472,0.018519
3,softmin_d1_beta20p0,129,0.847547,0.727547,0.018113,0.831698,0.704906,0.253585,0.041509,0.025660,0.844528,0.818868,1.378113,0.329057,0.025630
4,softmin_d1_beta0p1,150,0.859245,0.750943,0.023396,0.817736,0.686038,0.263396,0.050566,0.006792,0.821132,0.814340,1.363774,0.301887,0.020654


In [9]:
# Train the clean softmin d1 utility on all subsets of size <= 5.
# This keeps the optimized dense time x action training path; <=5 gives 637 static actions for 10 nodes.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import build_eq5_rssi_static_examples, run_rssi_utility_search

LEQ5_PREFIX = "m02_multi_vehicle_leq5_rssi_static"
LEQ5_REQUIRED = [
    PROJECT_ROOT / "data" / "processed" / f"{LEQ5_PREFIX}_arrays.npz",
    PROJECT_ROOT / "data" / "processed" / f"{LEQ5_PREFIX}_examples_index.csv",
    PROJECT_ROOT / "data" / "processed" / f"{LEQ5_PREFIX}_meta.json",
]

if all(path.exists() for path in LEQ5_REQUIRED):
    print(f"Using existing processed artifact: {LEQ5_PREFIX}")
else:
    leq5_summary_df, leq5_paths_df = build_eq5_rssi_static_examples(
        PROJECT_ROOT,
        prefix=LEQ5_PREFIX,
        exact_subset_size=5,
        subset_size_policy="leq",
    )
    display(leq5_summary_df)
    display(leq5_paths_df)

LEQ5_SOFTMIN_BETA5 = [
    {
        "name": "softmin_d1_beta5",
        "column": "derived_softmin_d1_beta5",
        "description": "Smooth min over per-vehicle d1 utilities with beta=5, trained over all subset sizes <= 5.",
        "kind": "softmin_d1",
        "beta": 5.0,
    }
]

leq5_softmin_summary_df, leq5_softmin_results = run_rssi_utility_search(
    PROJECT_ROOT,
    prefix=LEQ5_PREFIX,
    model_tag="multi_vehicle_rssi_softmin_d1_beta5_leq5_150ep",
    max_epochs=150,
    patience=150,
    log_every=5,
    batch_size=8192,
    time_batch_size=256,
    utility_candidates=LEQ5_SOFTMIN_BETA5,
)

vehicle_cols = [col for col in leq5_softmin_summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(leq5_softmin_summary_df[[col for col in display_cols if col in leq5_softmin_summary_df.columns]])


Interim source: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\interim\20250815_0900_multi_vehicle
valid timestamps=6625, nodes=10, vehicles=['vehicle1', 'vehicle2'], subset sizes <= 5
rho=10.222 m
Saved processed artifact: m02_multi_vehicle_leq5_rssi_static
Elapsed: 2.0 min


,prefix,times,actions_per_time,examples,context_dim,action_dim,mean_contains_fraction_all_actions,both_rate_all_actions,elapsed_min
0,m02_multi_vehicle_leq5_rssi_static,6625,637,4220125,30,21,0.401884,0.167433,2.008007


,artifact,path
0,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


Building dense train cache from m02_multi_vehicle_leq5_rssi_static_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_leq5_rssi_static_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=637, examples=4,220,125, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_leq5_rssi_static_dense_train_cache.npz
utilities: ['softmin_d1_beta5']

=== softmin_d1_beta5 === Smooth min over per-vehicle d1 utilities with beta=5, trained over all subset sizes <= 5.
softmin_d1_beta5 ep 001/150* loss=0.03755 val_acc=0.587 both=0.313 one=0.547 gap=0.038 vehicle1=0.568 vehicle2=0.606 wrank=2.35 reg=0.0738 2s
softmin_d1_beta5 ep 002/150* loss=0.01169 val_acc=0.640 both=0.395 one=0.491 gap=0.082 vehicle1=0.682 vehicle2=0.599 wrank=2.02 reg=0.05

,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,softmin_d1_beta5,117,0.756981,0.57434,0.021132,0.77283,0.621887,0.301887,0.076226,0.021132,0.762264,0.783396,1.535094,0.447547,0.034111


In [10]:
# Softmin over full per-vehicle 5-rank utilities, trained for both ==5 and <=5 action sets.
# Per vehicle: u_v = sum_j w_j / (1 + d_j/rho), then U = softmin_beta(u_1, u_2).
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import build_eq5_rssi_static_examples, run_rssi_utility_search

FULL5_WEIGHTS = [1.00, 0.45, 0.20, 0.10, 0.05]
FULL5_SOFTMIN_BETA5 = [
    {
        "name": "softmin_full5_beta5_w100_045_020_010_005",
        "column": "derived_softmin_full5_beta5_w100_045_020_010_005",
        "description": "softmin_beta5 over per-vehicle weighted rank utilities with weights [1,.45,.20,.10,.05].",
        "kind": "softmin_rank_weighted",
        "beta": 5.0,
        "weights": FULL5_WEIGHTS,
    }
]

FULL5_JOBS = [
    {
        "label": "eq5",
        "prefix": "m02_multi_vehicle_eq5_rssi_static_full5ranks",
        "subset_size_policy": "exact",
        "model_tag": "multi_vehicle_rssi_softmin_full5_eq5_150ep",
        "time_batch_size": 256,
    },
    {
        "label": "leq5",
        "prefix": "m02_multi_vehicle_leq5_rssi_static_full5ranks",
        "subset_size_policy": "leq",
        "model_tag": "multi_vehicle_rssi_softmin_full5_leq5_150ep",
        "time_batch_size": 128,
    },
]

full5_summaries = []
full5_results = {}
for job in FULL5_JOBS:
    prefix = job["prefix"]
    required = [
        PROJECT_ROOT / "data" / "processed" / f"{prefix}_arrays.npz",
        PROJECT_ROOT / "data" / "processed" / f"{prefix}_examples_index.csv",
        PROJECT_ROOT / "data" / "processed" / f"{prefix}_meta.json",
    ]
    if all(path.exists() for path in required):
        print(f"Using existing processed artifact: {prefix}")
    else:
        artifact_summary_df, artifact_paths_df = build_eq5_rssi_static_examples(
            PROJECT_ROOT,
            prefix=prefix,
            exact_subset_size=5,
            subset_size_policy=job["subset_size_policy"],
        )
        display(artifact_summary_df)
        display(artifact_paths_df)

    summary_df, results = run_rssi_utility_search(
        PROJECT_ROOT,
        prefix=prefix,
        model_tag=job["model_tag"],
        max_epochs=150,
        patience=150,
        log_every=5,
        batch_size=8192,
        time_batch_size=job["time_batch_size"],
        utility_candidates=FULL5_SOFTMIN_BETA5,
    )
    summary_df = summary_df.copy()
    summary_df.insert(0, "action_set", job["label"])
    full5_summaries.append(summary_df)
    full5_results[job["label"]] = results

full5_summary_df = pd.concat(full5_summaries, ignore_index=True)
vehicle_cols = [col for col in full5_summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "action_set",
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(full5_summary_df[[col for col in display_cols if col in full5_summary_df.columns]])


Interim source: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\interim\20250815_0900_multi_vehicle
valid timestamps=6625, nodes=10, vehicles=['vehicle1', 'vehicle2'], exact subset size=5
rho=10.222 m
Saved processed artifact: m02_multi_vehicle_eq5_rssi_static_full5ranks
Elapsed: 0.9 min


,prefix,times,actions_per_time,examples,context_dim,action_dim,mean_contains_fraction_all_actions,both_rate_all_actions,elapsed_min
0,m02_multi_vehicle_eq5_rssi_static_full5ranks,6625,252,1669500,30,21,0.5,0.245493,0.865493


,artifact,path
0,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


Building dense train cache from m02_multi_vehicle_eq5_rssi_static_full5ranks_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_full5ranks_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_full5ranks_dense_train_cache.npz
utilities: ['softmin_full5_beta5_w100_045_020_010_005']

=== softmin_full5_beta5_w100_045_020_010_005 === softmin_beta5 over per-vehicle weighted rank utilities with weights [1,.45,.20,.10,.05].
softmin_full5_beta5_w100_045_020_010_005 ep 001/150* loss=0.09535 val_acc=0.600 both=0.330 one=0.541 gap=0.002 vehicle1=0.602 vehicle2=0.599 wrank=2.28 reg=0.1087 1s
softmin_full5_beta5_w100_045_020_01

,prefix,times,actions_per_time,examples,context_dim,action_dim,mean_contains_fraction_all_actions,both_rate_all_actions,elapsed_min
0,m02_multi_vehicle_leq5_rssi_static_full5ranks,6625,637,4220125,30,21,0.401884,0.167433,2.2433


,artifact,path
0,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


Building dense train cache from m02_multi_vehicle_leq5_rssi_static_full5ranks_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_leq5_rssi_static_full5ranks_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=637, examples=4,220,125, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_leq5_rssi_static_full5ranks_dense_train_cache.npz
utilities: ['softmin_full5_beta5_w100_045_020_010_005']

=== softmin_full5_beta5_w100_045_020_010_005 === softmin_beta5 over per-vehicle weighted rank utilities with weights [1,.45,.20,.10,.05].
softmin_full5_beta5_w100_045_020_010_005 ep 001/150* loss=0.05062 val_acc=0.641 both=0.377 one=0.528 gap=0.019 vehicle1=0.651 vehicle2=0.632 wrank=2.09 reg=0.1022 2s
softmin_full5_beta5_w100_045_020

,action_set,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,eq5,softmin_full5_beta5_w100_045_020_010_005,132,0.895472,0.795472,0.002264,0.860377,0.744151,0.232453,0.023396,0.063396,0.892075,0.828679,1.350943,0.326792,0.040712
1,leq5,softmin_full5_beta5_w100_045_020_010_005,149,0.870943,0.755472,0.025660,0.838868,0.698868,0.280000,0.021132,0.036981,0.857359,0.820377,1.378113,0.356981,0.044178


In [11]:
# Winning config with v3 refined-band context variants: v3 bands only vs RSSI + v3 bands.
# Keeps exact-5 actions, static action geometry, v3 architecture, and softmin_d1_beta5 utility.
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import build_eq5_rssi_static_examples, run_rssi_utility_search

V3_CONTEXT_AUDIO_FEATURES = list(mv_utils.V3_REFINED_BAND_DB_FEATURES)
print("V3 context audio features:", V3_CONTEXT_AUDIO_FEATURES)

WINNING_SOFTMIN_D1_BETA5 = [
    {
        "name": "softmin_d1_beta5",
        "column": "derived_softmin_d1_beta5",
        "description": "Smooth min over per-vehicle d1 utilities with beta=5.",
        "kind": "softmin_d1",
        "beta": 5.0,
    }
]

FEATURE_CONTEXT_JOBS = [
    {
        "label": "v3_feature_bands",
        "prefix": "m02_multi_vehicle_eq5_v3_feature_bands_static",
        "context_feature_mode": "feature_bands",
        "model_tag": "multi_vehicle_eq5_softmin_d1_beta5_v3_feature_bands_150ep",
    },
    {
        "label": "rssi_plus_v3_feature_bands",
        "prefix": "m02_multi_vehicle_eq5_rssi_plus_v3_feature_bands_static",
        "context_feature_mode": "rssi_plus_feature_bands",
        "model_tag": "multi_vehicle_eq5_softmin_d1_beta5_rssi_plus_v3_feature_bands_150ep",
    },
]

feature_context_summaries = []
feature_context_results = {}
for job in FEATURE_CONTEXT_JOBS:
    prefix = job["prefix"]
    required = [
        PROJECT_ROOT / "data" / "processed" / f"{prefix}_arrays.npz",
        PROJECT_ROOT / "data" / "processed" / f"{prefix}_examples_index.csv",
        PROJECT_ROOT / "data" / "processed" / f"{prefix}_meta.json",
    ]
    if all(path.exists() for path in required):
        print(f"Using existing processed artifact: {prefix}")
    else:
        artifact_summary_df, artifact_paths_df = build_eq5_rssi_static_examples(
            PROJECT_ROOT,
            prefix=prefix,
            exact_subset_size=5,
            subset_size_policy="exact",
            context_feature_mode=job["context_feature_mode"],
            audio_band_features=V3_CONTEXT_AUDIO_FEATURES,
        )
        display(artifact_summary_df)
        display(artifact_paths_df)

    summary_df, results = run_rssi_utility_search(
        PROJECT_ROOT,
        prefix=prefix,
        model_tag=job["model_tag"],
        max_epochs=150,
        patience=150,
        log_every=5,
        batch_size=8192,
        time_batch_size=256,
        utility_candidates=WINNING_SOFTMIN_D1_BETA5,
    )
    summary_df = summary_df.copy()
    summary_df.insert(0, "context_mode", job["label"])
    feature_context_summaries.append(summary_df)
    feature_context_results[job["label"]] = results

feature_context_summary_df = pd.concat(feature_context_summaries, ignore_index=True)
vehicle_cols = [col for col in feature_context_summary_df.columns if col.startswith("test_contains_vehicle")]
display_cols = [
    "context_mode",
    "utility",
    "best_epoch",
    "val_contains_closest_fraction",
    "val_contains_closest_all",
    "val_dominance_gap",
    "test_contains_closest_fraction",
    "test_contains_closest_all",
    "test_one_closest_rate",
    "test_neither_closest_rate",
    "test_dominance_gap",
    *vehicle_cols,
    "test_mean_worst_vehicle_rank",
    "test_mean_abs_vehicle_rank_gap",
    "test_avg_regret",
]
display(feature_context_summary_df[[col for col in display_cols if col in feature_context_summary_df.columns]])


Interim source: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\interim\20250815_0900_multi_vehicle
valid timestamps=6625, nodes=10, vehicles=['vehicle1', 'vehicle2'], exact subset size=5
rho=10.222 m
Saved processed artifact: m02_multi_vehicle_eq5_feature_bands_static
Elapsed: 0.9 min


,prefix,times,actions_per_time,examples,context_dim,action_dim,mean_contains_fraction_all_actions,both_rate_all_actions,elapsed_min
0,m02_multi_vehicle_eq5_feature_bands_static,6625,252,1669500,60,21,0.5,0.245493,0.898481


,artifact,path
0,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


Building dense train cache from m02_multi_vehicle_eq5_feature_bands_static_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_feature_bands_static_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=60, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_feature_bands_static_dense_train_cache.npz
utilities: ['softmin_d1_beta5']

=== softmin_d1_beta5 === Smooth min over per-vehicle d1 utilities with beta=5.
softmin_d1_beta5 ep 001/150* loss=0.04828 val_acc=0.583 both=0.298 one=0.571 gap=0.048 vehicle1=0.559 vehicle2=0.608 wrank=2.09 reg=0.0803 1s
softmin_d1_beta5 ep 003/150* loss=0.00888 val_acc=0.674 both=0.448 one=0.451 gap=0.028 vehicle1=0.660 vehicle2=0.688 wrank=1.74 reg=0.0470 2s
softmi

,prefix,times,actions_per_time,examples,context_dim,action_dim,mean_contains_fraction_all_actions,both_rate_all_actions,elapsed_min
0,m02_multi_vehicle_eq5_rssi_plus_feature_bands_...,6625,252,1669500,70,21,0.5,0.245493,0.999383


,artifact,path
0,arrays_npz,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
1,examples_index_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
2,feature_wide_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
3,sensor_geometry_csv,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...
4,meta_json,d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-...


Building dense train cache from m02_multi_vehicle_eq5_rssi_plus_feature_bands_static_examples_index.csv ...
Saved dense train cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_plus_feature_bands_static_dense_train_cache.npz
PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=70, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_plus_feature_bands_static_dense_train_cache.npz
utilities: ['softmin_d1_beta5']

=== softmin_d1_beta5 === Smooth min over per-vehicle d1 utilities with beta=5.
softmin_d1_beta5 ep 001/150* loss=0.03902 val_acc=0.569 both=0.316 one=0.506 gap=0.023 vehicle1=0.558 vehicle2=0.580 wrank=2.18 reg=0.0757 1s
softmin_d1_beta5 ep 002/150* loss=0.01131 val_acc=0.687 both=0.471 one=0.432 gap=0.023 vehicle1=0.698 vehicle2=0.675 w

,context_mode,utility,best_epoch,val_contains_closest_fraction,val_contains_closest_all,val_dominance_gap,test_contains_closest_fraction,test_contains_closest_all,test_one_closest_rate,test_neither_closest_rate,test_dominance_gap,test_contains_vehicle1,test_contains_vehicle2,test_mean_worst_vehicle_rank,test_mean_abs_vehicle_rank_gap,test_avg_regret
0,feature_bands,softmin_d1_beta5,137,0.844906,0.713962,0.067170,0.864906,0.750189,0.229434,0.020377,0.034717,0.882264,0.847547,1.33434,0.313962,0.019553
1,rssi_plus_feature_bands,softmin_d1_beta5,76,0.827925,0.672453,0.045283,0.834340,0.689811,0.289057,0.021132,0.003774,0.836226,0.832453,1.37283,0.351698,0.025129


In [12]:
# Final RSSI-only exact-5 model: best_model_m_v1
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import importlib
import multi_vehicle_utility_search as mv_utils

mv_utils = importlib.reload(mv_utils)
from multi_vehicle_utility_search import run_rssi_utility_search

BEST_MODEL_M_V1_PREFIX = "m02_multi_vehicle_eq5_rssi_static"
BEST_MODEL_M_V1_TAG = "best_model_m_v1"

BEST_MODEL_M_V1_UTILITY = [
    {
        "name": "softmin_d1_beta5",
        "column": "derived_softmin_d1_beta5",
        "description": "Smooth min over per-vehicle d1 utilities with beta=5.",
        "kind": "softmin_d1",
        "beta": 5.0,
    }
]

best_model_m_v1_summary_df, best_model_m_v1_results = run_rssi_utility_search(
    PROJECT_ROOT,
    prefix=BEST_MODEL_M_V1_PREFIX,
    model_tag=BEST_MODEL_M_V1_TAG,
    max_epochs=400,
    patience=400,
    log_every=10,
    batch_size=8192,
    time_batch_size=256,
    utility_candidates=BEST_MODEL_M_V1_UTILITY,
)

display(best_model_m_v1_summary_df)

print("Checkpoint:", PROJECT_ROOT / "experiments" / BEST_MODEL_M_V1_TAG / "softmin_d1_beta5" / "best_model.pt")
print("Tables:", PROJECT_ROOT / "reports" / "tables" / BEST_MODEL_M_V1_TAG)


PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt
DEVICE: cuda
data: times=6625, actions/time=252, examples=1,669,500, context_dim=30, action_dim=21
dense cache: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\data\processed\m02_multi_vehicle_eq5_rssi_static_dense_train_cache.npz
utilities: ['softmin_d1_beta5']

=== softmin_d1_beta5 === Smooth min over per-vehicle d1 utilities with beta=5.
softmin_d1_beta5 ep 001/400* loss=0.04217 val_acc=0.601 both=0.326 one=0.550 gap=0.022 vehicle1=0.612 vehicle2=0.590 wrank=2.30 reg=0.0678 1s
softmin_d1_beta5 ep 002/400* loss=0.01223 val_acc=0.651 both=0.398 one=0.505 gap=0.070 vehicle1=0.686 vehicle2=0.616 wrank=1.96 reg=0.0546 1s
softmin_d1_beta5 ep 004/400* loss=0.00809 val_acc=0.661 both=0.454 one=0.414 gap=0.011 vehicle1=0.666 vehicle2=0.656 wrank=1.75 reg=0.0453 2s
softmin_d1_beta5 ep 005/400* loss=0.00744 val_acc=0.682 both=0.489 one=0.385 gap=0.002 vehicle1=0.682 vehicle2=0.681 wrank=1.65 reg=0.0408 

,utility,utility_column,best_epoch,train_rmse,train_mae,train_r2,train_top1,train_top3,train_mean_rank,train_avg_regret,...,test_mean_rank_vehicle1,test_mean_d1_vehicle1,test_contains_vehicle2,test_mean_rank_vehicle2,test_mean_d1_vehicle2,test_dominance_gap,test_vehicle1_only_rate,test_vehicle2_only_rate,test_mean_abs_vehicle_rank_gap,test_mean_abs_d1_gap_m
0,softmin_d1_beta5,derived_softmin_d1_beta5,111,0.041381,0.030857,0.835095,0.951447,0.951447,4.096352,0.001325,...,1.147925,12.925476,0.868679,1.149434,17.651718,0.000755,0.116226,0.115472,0.26717,6.535304


Checkpoint: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\experiments\best_model_m_v1\softmin_d1_beta5\best_model.pt
Tables: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\reports\tables\best_model_m_v1


In [13]:
# Display best_model_m_v1 test metrics clearly
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_TAG = "best_model_m_v1"
UTILITY_NAME = "softmin_d1_beta5"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / MODEL_TAG
METRICS_PATH = TABLE_DIR / f"{UTILITY_NAME}_metrics.csv"
SUMMARY_PATH = TABLE_DIR / "utility_search_summary.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "experiments" / MODEL_TAG / UTILITY_NAME / "best_model.pt"

if METRICS_PATH.exists():
    all_metrics = pd.read_csv(METRICS_PATH)
    test_metrics = all_metrics.loc[all_metrics["split"].eq("test")].copy()
elif SUMMARY_PATH.exists():
    summary = pd.read_csv(SUMMARY_PATH)
    test_cols = [col for col in summary.columns if col.startswith("test_")]
    test_metrics = summary[["utility", "best_epoch", *test_cols]].rename(
        columns={col: col.replace("test_", "") for col in test_cols}
    )
    test_metrics["split"] = "test"
else:
    raise FileNotFoundError(f"Could not find saved metrics under {TABLE_DIR}")

row = test_metrics.iloc[0]
score_from_parts = row["contains_closest_all"] + 0.5 * row["one_closest_rate"]

m_v1_test_report = pd.DataFrame(
    [
        {
            "model": MODEL_TAG,
            "utility": row["utility"],
            "best_epoch": int(row["best_epoch"]),
            "test_timestamps": int(row["n_timestamps"]),
            "containment_score_%": 100.0 * row["contains_closest_fraction"],
            "both_closest_%": 100.0 * row["contains_closest_all"],
            "one_closest_%": 100.0 * row["one_closest_rate"],
            "neither_closest_%": 100.0 * row["neither_closest_rate"],
            "vehicle1_contained_%": 100.0 * row["contains_vehicle1"],
            "vehicle2_contained_%": 100.0 * row["contains_vehicle2"],
            "dominance_gap_pp": 100.0 * row["dominance_gap"],
            "vehicle1_only_%": 100.0 * row["vehicle1_only_rate"],
            "vehicle2_only_%": 100.0 * row["vehicle2_only_rate"],
            "mean_worst_rank": row["mean_worst_vehicle_rank"],
            "mean_rank_vehicle1": row["mean_rank_vehicle1"],
            "mean_rank_vehicle2": row["mean_rank_vehicle2"],
            "mean_d1_vehicle1_m": row["mean_d1_vehicle1"],
            "mean_d1_vehicle2_m": row["mean_d1_vehicle2"],
            "avg_regret": row["avg_regret"],
            "rmse": row["rmse"],
            "score_check_abs_diff": abs(row["contains_closest_fraction"] - score_from_parts),
        }
    ]
)

percent_cols = [col for col in m_v1_test_report.columns if col.endswith("_%") or col.endswith("_pp")]
rank_cols = ["mean_worst_rank", "mean_rank_vehicle1", "mean_rank_vehicle2"]
distance_cols = ["mean_d1_vehicle1_m", "mean_d1_vehicle2_m"]
rounding = {col: 2 for col in percent_cols}
rounding.update({col: 3 for col in rank_cols})
rounding.update({col: 2 for col in distance_cols})
rounding.update({"avg_regret": 5, "rmse": 5, "score_check_abs_diff": 8})

print(f"Metrics: {METRICS_PATH if METRICS_PATH.exists() else SUMMARY_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print("containment_score = both_closest + 0.5 * one_closest")
display(m_v1_test_report.round(rounding))


Metrics: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\reports\tables\best_model_m_v1\softmin_d1_beta5_metrics.csv
Checkpoint: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26-MILCOMAttempt\experiments\best_model_m_v1\softmin_d1_beta5\best_model.pt
containment_score = both_closest + 0.5 * one_closest


,model,utility,best_epoch,test_timestamps,containment_score_%,both_closest_%,one_closest_%,neither_closest_%,vehicle1_contained_%,vehicle2_contained_%,...,vehicle1_only_%,vehicle2_only_%,mean_worst_rank,mean_rank_vehicle1,mean_rank_vehicle2,mean_d1_vehicle1_m,mean_d1_vehicle2_m,avg_regret,rmse,score_check_abs_diff
0,best_model_m_v1,softmin_d1_beta5,111,1325,86.91,75.32,23.17,1.51,86.94,86.87,...,11.62,11.55,1.282,1.148,1.149,12.93,17.65,0.01832,0.06456,4.000000e-08
